# VFX Production Intelligence Dashboard — Reviews Cleaning

Work through this table from top to bottom. The context and rules below are inherited from earlier milestones.

## What we already know

- **Business name:** Reviews
- **Purpose:** Stores shot-review events used to analyze revision activity, approval outcomes, internal-versus-client feedback, review turnaround time, and review-related production risk.
- **One row represents:** One review event for one shot.
- **Expected primary key:** `review_id`

### Relationships
- shot_id → shots.shot_id
- project_id → projects.project_id

### Field rules from the approved data dictionary

| Field | Definition | Expected type | Null rule | Key role | Uniqueness | Allowed values / format | Cleaning expectation |
|---|---|---|---|---|---|---|---|
| `review_id` | Unique identifier assigned to each recorded shot-review event. | Identifier text | No nulls allowed | Primary key | Required | REV-#######: uppercase REV, a hyphen, and seven digits. | Remove exact duplicate review rows, quarantine conflicting repeated IDs, and verify that the cleaned key is unique and non-null. |
| `shot_id` | Identifier of the production shot evaluated during the review event. | Identifier text | No nulls allowed | Foreign key | Not required | PRJ-####_SQ###_SH####. | Trim and uppercase valid IDs, correct only confidently recoverable values, and quarantine null, placeholder, or unresolved references. |
| `project_id` | Identifier of the production project associated with the reviewed shot. | Identifier text | No nulls allowed | Foreign key | Not required | PRJ-####. | Trim and uppercase IDs, validate them against projects.project_id, reconcile each value to the related shot’s project, and quarantine unresolved contradictions. |
| `review_date` | Calendar date on which review feedback was recorded. | Date | No nulls allowed | Not a key | Not required | YYYY-MM-DD. | Parse valid formats, store as a true date, and quarantine blank, placeholder, or impossible values that cannot be corrected reliably. |
| `review_round` | Sequential number representing the review round reached by the shot. | Integer | No nulls allowed | Not a key | Not required | Positive whole number beginning at 1. | Convert confidently interpretable values to positive integers, validate sequence within each shot, and quarantine unresolved or non-positive values. |
| `review_type` | Indicates whether the review was conducted internally or by a client-side reviewer. | Category text | No nulls allowed | Not a key | Not required | Internal Review; Client Review. | Trim whitespace and standardize to the two approved review types. |
| `feedback_source` | Role or stakeholder group responsible for supplying the review feedback. | Category text | No nulls allowed | Not a key | Not required | Department Lead; Comp Supervisor; VFX Supervisor; Client Supervisor; Agency Creative. | Trim whitespace and standardize capitalization without combining genuinely different feedback-source roles. |
| `outcome` | Result assigned to the review event, indicating acceptance, requested changes, notes, or a hold. | Category text | No nulls allowed | Not a key | Not required | Approved; Approved with Notes; Changes Requested; Minor Notes; Hold. | Trim whitespace and map known variants to the five approved outcomes; investigate unfamiliar values before mapping. |
| `note_category` | Primary category describing the type of feedback or correction recorded during the review. | Category text | No nulls allowed | Not a key | Not required | Color; Technical; Integration; Client Preference; Timing; Editorial Change; Edge Work; Continuity. | Trim whitespace and standardize to the approved feedback categories; validate logical consistency with the review outcome. |
| `response_hours` | Number of elapsed hours until review feedback was acknowledged or addressed. | Decimal | No nulls allowed | Not a key | Not required | Non-negative decimal value. | Convert valid values to decimal, reject negatives, investigate blanks and extreme values against review context and client SLA, and exclude unresolved values from averages. |
| `reviewer` | Name of the individual recorded as conducting or communicating the review. | Text | No nulls allowed | Not a key | Not required | Recorded person name, either full name or initial and surname. | Trim whitespace and standardize display formatting where the same person can be identified confidently. |
| `notes` | Optional free-form text providing additional context about review feedback or requested changes. | Text | Nulls allowed | Not a key | Not required | Free-form text, or null when no additional comment is recorded. | Trim populated text, convert empty strings to null, and preserve the original wording. |

### Known issues and approved decisions
- review_id: The raw table contains 3,883 rows and 3,843 distinct review IDs, producing 40 repeated rows.
- review_id: review_id remains the intended primary key. Repeated IDs are raw-data defects rather than a change to the event grain.
- shot_id: The raw table has nine null shot IDs and 26 non-null values that do not match a shot, including placeholder and case-format issues.
- shot_id: shot_id remains a required foreign key; unresolved review-to-shot references must be excluded from joined review analysis.
- project_id: The raw table contains project-key issues and some project IDs disagree with the project derived from shot_id.
- project_id: project_id remains a required foreign key and must agree with both the Projects table and the project inherited through shot_id.
- review_date: The raw text field contains 72 mixed, blank, placeholder, or invalid date issues, including TBD and impossible dates.
- review_date: The field is logically a required date; malformed source values explain the text type and require cleaning.
- review_round: The raw text field includes Round 2, 03, first, blanks, and -1.
- review_round: The field is expected to be an integer; malformed source labels and invalid values explain the observed text type.
- feedback_source: Repeated values are expected because the same role category applies to many review events.
- outcome: The raw field contains 45 inconsistencies, including approved, CHANGE REQUEST, Minor notes with trailing whitespace, and Hold/Waiting.
- outcome: Multiple raw labels represent the same review outcomes and will be standardized for consistent approval and revision metrics.
- response_hours: The raw text field contains blanks, pending, negative values, and extreme values such as 240 hours.
- response_hours: The field is expected to be numeric and non-negative; malformed and extreme source values explain the text type and require remediation.
- reviewer: Repeated reviewer names are expected because one person may review many shots.
- notes: Missing notes and repeated common instructions are expected and do not by themselves indicate duplicate reviews.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r'C:/Users/Dan/Documents/MEGA/Dev/GitHub/career-accelerator/projects/project-01-vfx-production-intelligence')
TABLE_NAME = 'reviews'
RAW_PATH = PROJECT_DIR / r'data/raw/csv/raw_reviews.csv'
PROCESSED_PATH = PROJECT_DIR / r'data/processed/csv/reviews.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

if RAW_PATH.suffix.lower() == '.csv':
    raw_df = pd.read_csv(RAW_PATH)
elif RAW_PATH.suffix.lower() == '.parquet':
    raw_df = pd.read_parquet(RAW_PATH)
else:
    raise ValueError(f'Add the appropriate pandas reader for {RAW_PATH.suffix}')

clean_df = raw_df.copy()
print(f'{TABLE_NAME}: {len(raw_df):,} raw rows, {len(raw_df.columns)} columns')
raw_df.head()


## 1. Profile the raw table

- Confirm the source row count and column names.
- Measure missing values by field.
- Check exact duplicate rows.
- Test uniqueness and nulls for `review_id`.
- Review observed categories and parsing problems.
- Compare findings with the dictionary rules above before changing data.

In [ ]:
# Write the profiling checks for this table here.
# Keep the outputs that justify your cleaning decisions.


## 2. Apply the approved cleaning plan

Transform `clean_df` without modifying `raw_df`. Follow the field-level expectations above. Document any treatment that differs from the approved dictionary.

In [ ]:
# Write this table's cleaning transformations here.
# Example structure only: clean_df = clean_df.copy()


## 3. Validate the processed result

- Required columns are still present.
- Expected logical types can be produced consistently.
- Required fields do not contain unresolved nulls.
- Allowed values and formats match the dictionary.
- Invalid negative, out-of-range, or impossible values are resolved or documented.
- `review_id` is non-null and unique.
- Foreign-key and relationship exceptions are measured and documented.

In [ ]:
# Write the before-and-after validation checks here.
# The checks should fail visibly when an unresolved issue remains.


## 4. Export the reviewed table

After validation, save the reviewed result to `data/processed/csv/reviews.csv`. The Data Cleaning Studio will discover and validate the file.

In [ ]:
# Run only after the table has passed your validation checks.
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved {len(clean_df):,} rows to {PROCESSED_PATH}')


## Cleaning summary

<!-- Describe what changed, why each important decision was appropriate, how many records were affected, and any remaining exception that a later milestone must know about. -->
